GPU details

In [ ]:
!nvidia-smi

Thu Aug  6 15:21:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Installing Libraries

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.2 MB/s eta 0:00:00


In [ ]:
import torch, pandas as pd, json, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata
import os

login(userdata.get('HF_TOKEN'))

Model and it's configurations

In [ ]:
MODEL_ID = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("loaded")

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

loaded


In [ ]:
def build_eval_prompt(row):
    """Prompts the model to answer questions and respond with a single letter"""
    return (f"Answer this medical multiple-choice question with a single letter.\n\n"
            f"Question: {row['question']}\n"
            f"A) {row['opa']}\nB) {row['opb']}\nC) {row['opc']}\nD) {row['opd']}\n\n"
            f"Answer:")

def to_chat(prompt):
    msgs = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)

In [ ]:
def strip_thought_tags(text):
    """Gemma 4 emits <|channel>thought ... <channel|> blocks; remove them."""
    return re.sub(r'<\|channel\>thought.*?\<channel\|\>', '', text, flags=re.S)

def parse_letter(text):
    cleaned = strip_thought_tags(text)
    m = re.search(r'\b([A-D])\b', cleaned.upper())
    return m.group(1) if m else None

@torch.no_grad()
def answer_question(row, max_new_tokens=20):
    chat = to_chat(build_eval_prompt(row))
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False,                 # deterministic
                         pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=False)
    return gen, parse_letter(gen)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Research Project Synthetic Data'
os.makedirs(f'{BASE}/generated', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Testing pipeline on 5 questions

In [ ]:
valid = pd.read_json(f'{BASE}/data/dev.json', lines=True)
valid['subject_name'] = valid['subject_name'].str.strip().str.lower()

for _, row in valid.head(5).iterrows():
    raw, pred = answer_question(row)
    true = 'ABCD'[int(row['cop']) - 1]
    print(f"RAW: {repr(raw[:200])}")
    print(f"  pred={pred} true={true} correct={pred == true}\n")

RAW: 'A<turn|>'
  pred=A true=A correct=True

RAW: 'A<turn|>'
  pred=A true=A correct=True

RAW: 'C<turn|>'
  pred=C true=C correct=True

RAW: 'C<turn|>'
  pred=C true=C correct=True

RAW: 'B<turn|>'
  pred=B true=A correct=False



In [ ]:
from tqdm import tqdm

# using the same 16 subjects as the taxonomy
with open(f'{BASE}/Taxonomy/taxonomy_filtered.json') as f:
    taxonomy = json.load(f)
subjects = list(taxonomy.keys())

eval_set = valid[valid['subject_name'].isin(subjects)].copy()
print(f"Evaluating on {len(eval_set)} questions across {len(subjects)} subjects")

results = []
for _, row in tqdm(eval_set.iterrows(), total=len(eval_set)):
    raw, pred = answer_question(row)
    results.append({
        "subject": row['subject_name'],
        "predicted": pred,
        "true": 'ABCD'[int(row['cop']) - 1],
        "correct": pred == 'ABCD'[int(row['cop']) - 1],
        "raw": raw
    })

res_df = pd.DataFrame(results)
res_df.to_csv(f'{BASE}/baseline_eval.csv', index=False)

print(f"\nBASELINE ACCURACY: {res_df['correct'].mean():.1%}")
print(f"Unparseable: {res_df['predicted'].isna().sum()}")
print("\nPer subject:")
print(res_df.groupby('subject')['correct'].agg(['mean', 'size'])
      .sort_values('mean', ascending=False).to_string())

Evaluating on 4094 questions across 16 subjects


100%|██████████| 4094/4094 [21:53<00:00,  3.12it/s]



BASELINE ACCURACY: 51.8%
Unparseable: 1

Per subject:
                                  mean  size
subject                                     
pharmacology                  0.637860   243
biochemistry                  0.614035   171
ent                           0.603774    53
medicine                      0.583051   295
pathology                     0.581602   337
radiology                     0.579710    69
physiology                    0.573099   171
anatomy                       0.572650   234
microbiology                  0.516393   122
ophthalmology                 0.482759    58
surgery                       0.476965   369
gynaecology & obstetrics      0.473214   224
social & preventive medicine  0.472868   129
pediatrics                    0.470085   234
dental                        0.466616  1318
forensic medicine             0.462687    67
